In [1]:
import requests
import pandas

In [12]:
import requests
import csv
import pandas as pd
from io import StringIO, BytesIO
import gzip

# STEP 1: Download CSV & Extract Last Names
def get_last_names_from_csv():

    url = 'https://docs.google.com/spreadsheets/d/1yQ-LfAp9KOCq74p9xWGBWNt4fUvsMctUy9bLYCgvrJI/export?format=csv'
    response = requests.get(url)
    response.raise_for_status()
    last_names = pd.read_csv(StringIO(response.text))
    # print(tabulate(last_names.head(), headers='keys', tablefmt='rounded_grid'))
    last_name_list = 'Full List'
    last_names_to_itr = last_names[last_name_list][last_names[last_name_list].notna()]
    return last_names_to_itr

# STEP 2: Create API Payload
def create_payload(last_name):
    return f'7|0|9|https://wipp.edmundsassoc.com/Wipp/wipp/|6097249917502855D47FCE321E6AF9E1|wipp.client.WippService|taxOwnerNameSearch|java.lang.String/2004016611|Z|23|{last_name}||1|2|3|4|4|5|5|5|6|7|8|9|1|'

# STEP 3: Send API Request
def fetch_data(last_name):
    url = "https://wipp.edmundsassoc.com/Wipp/wipp/wipp"
    headers = {
        "Content-Type": "text/x-gwt-rpc; charset=UTF-8",
        "User-Agent": "Mozilla/5.0",
        "Origin": "https://wipp.edmundsassoc.com",
        "Referer": "https://wipp.edmundsassoc.com/Wipp/?wippid=23",
        "X-GWT-Module-Base": "https://wipp.edmundsassoc.com/Wipp/wipp/",
        "X-GWT-Permutation": "6097249917502855D47FCE321E6AF9E1",
        "Accept-Encoding": "gzip, deflate, br"
    }
    
    payload = create_payload(last_name)
    response = requests.post(url, data=payload, headers=headers)
    
    if response.status_code == 200:
        return response.text
    else:
        print(f"⚠️ Error fetching {last_name}: {response.status_code}")
        return None

# STEP 4: Parse API Response
import re

def parse_response(response, last_name):
    try:
        # Extract the list part (inside square brackets)
        match = re.search(r'\["java.util.ArrayList/4159755760","java.lang.String/2004016611",(.*)]', response)
        if not match:
            print(f"⚠️ No valid data found for {last_name}")
            return []
        
        raw_data = match.group(1)
        
        # Use a regex pattern to correctly extract quoted values, handling embedded commas
        pattern = r'"([^"]*?)"'
        extracted_data = re.findall(pattern, raw_data)

        # Group data into (Full Name, Address, Extra Info)
        parsed_data = []
        for i in range(0, len(extracted_data), 3):
            if i + 2 < len(extracted_data):
                full_name = extracted_data[i].strip().replace('\\x26','')
                address = extracted_data[i + 1].strip()
                extra_info = extracted_data[i + 2].strip()
                if last_name.lower() in full_name.lower():
                    parsed_data.append([last_name, full_name, address, "Middletown"])

        return parsed_data

    except Exception as e:
        print(f"⚠️ Error parsing response for {last_name}: {e}")
        return []


# STEP 5: Process Data & Create DataFrame
def main():
    last_names = get_last_names_from_csv()
    all_results = []

    for last_name in last_names:
        if last_name.strip():
            print(f"🔍 Fetching data for {last_name}...")
            response = fetch_data(last_name)
            if response:
                all_results.extend(parse_response(response, last_name))

    # Convert to DataFrame
    df = pd.DataFrame(all_results, columns=["Last Name", "Full Name", "Address", "Township"])
    
    # Save to CSV
    df.to_csv("Middletown.csv", index=False)
    print("✅ Done! Data saved to output.csv")
    #print(df.head())

# Run the script
main()


🔍 Fetching data for Adajania...
🔍 Fetching data for Adani...
🔍 Fetching data for Ambani...
🔍 Fetching data for Amin...
🔍 Fetching data for Antala...
🔍 Fetching data for Barot...
🔍 Fetching data for Bhagat...
🔍 Fetching data for Bharucha...
🔍 Fetching data for Bhatt...
🔍 Fetching data for Bhavasar...
🔍 Fetching data for Bhavsar...
🔍 Fetching data for Brahmbhatt...
🔍 Fetching data for But...
🔍 Fetching data for Chauhan...
🔍 Fetching data for Chavda...
🔍 Fetching data for Chokshi ...
🔍 Fetching data for Choksi...
🔍 Fetching data for Dalal...
🔍 Fetching data for Dani...
🔍 Fetching data for Darji...
🔍 Fetching data for Daruwala...
🔍 Fetching data for Dave...
🔍 Fetching data for Desai...
🔍 Fetching data for Dholakia...
🔍 Fetching data for Divatia...
🔍 Fetching data for Doshi...
🔍 Fetching data for Faldu...
🔍 Fetching data for Gadhavi...
🔍 Fetching data for Gajjar...
🔍 Fetching data for Ganatra...
🔍 Fetching data for Gandhi...
🔍 Fetching data for Goswami...
🔍 Fetching data for Jain...
🔍 Fetch